In [1]:
# ==============================================================
# 🧠 Local sanity test: Train PINN (data-only) on one dataset
# ==============================================================
import os
import torch
import pickle
import numpy as np
from omegaconf import OmegaConf
from src.nn.nn_actions import NeuralNetworkActions
from src.ode.sm_models_d import SynchronousMachineModels
from src.nn.nn_dataset import DataSampler
import torch.optim as optim


def load_dataset(dataset_path, device):
    with open(dataset_path, "rb") as f:
        dataset = pickle.load(f)

    x_all, y_all = [], []
    for r in dataset:
        t = np.array(r[0])
        y = np.vstack(r[1:]).T
        x = np.hstack([t.reshape(-1, 1), y])
        x_all.append(x)
        y_all.append(y)

    x_all = np.vstack(x_all)
    y_all = np.vstack(y_all)
    x_tensor = torch.tensor(x_all, dtype=torch.float32, device=device, requires_grad=True)
    y_tensor = torch.tensor(y_all, dtype=torch.float32, device=device)
    return x_tensor, y_tensor


def train_single_pinn(cfg, dataset_name="set5_mixed"):
    dataset_path = f"data/SM_AVR_GOV/dataset_{dataset_name}.pkl"
    model_save_path = f"model/SM_AVR_GOV/pinn_{dataset_name}_dataonly_test.pth"

    print(f"\n🚀 Training PINN on dataset: {dataset_path}")

    # --- Load dataset first ---
    ds = DataSampler(cfg, dataset_path=dataset_path)

    # --- Initialize model + network with preloaded data loader ---
    modelling_full = SynchronousMachineModels(cfg)
    network = NeuralNetworkActions(cfg, modelling_full, data_loader=ds)

    # Force data-only loss
    cfg.nn.weighting.weights = [1.0, 0.0, 0.0, 0.0]
    network.weight_data, network.weight_dt, network.weight_pinn, network.weight_pinn_ic = 1.0, 0.0, 0.0, 0.0

    # --- Split data ---
    x_train, y_train, x_col, x_ic, y_ic, x_val, y_val = ds.define_train_val_data2(
        cfg.dataset.perc_of_data_points,
        cfg.dataset.perc_of_col_points,
        1, 1, 1
    )

    device = network.device
    x_train = x_train.to(device).clone().detach().requires_grad_(True)
    y_train = y_train.to(device)
    x_val = x_val.to(device).clone().detach().requires_grad_(True)
    y_val = y_val.to(device)

    # --- Optimizer ---
    network.optimizer = optim.LBFGS(
        network.model.parameters(),
        lr=cfg.nn.lr,
        line_search_fn="strong_wolfe"
    )

    # --- Training loop ---
    print(f"Training {cfg.nn.type} for {cfg.nn.num_epochs} epochs (data-only)...")
    val_losses = []

    for epoch in range(cfg.nn.num_epochs):
        network.model.train()

        def closure():
            network.optimizer.zero_grad(set_to_none=True)
            y_hat, _, _ = network.calculate_point_grad2(x_train.clone().detach().requires_grad_(True), y_train)
            loss = network.criterion(y_hat, y_train)
            loss.backward()
            return loss

        loss = network.optimizer.step(closure)

        if (epoch + 1) % 10 == 0:
            network.model.eval()
            with torch.no_grad():
                y_val_pred = network.forward_pass(x_val)
                val_loss = network.criterion(y_val_pred, y_val).item()
                val_losses.append(val_loss)
            print(f"Epoch {epoch+1}/{cfg.nn.num_epochs} | Train Loss: {loss.item():.3e} | Val Loss: {val_loss:.3e}")

    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
    torch.save({
        "model_state_dict": network.model.state_dict(),
        "config": OmegaConf.to_container(cfg, resolve=True),
        "val_losses": val_losses,
    }, model_save_path)

    print(f"✅ Saved model to: {model_save_path}\n")
    print(f"📉 Final validation loss: {val_losses[-1]:.3e}")


if __name__ == "__main__":
    cfg = OmegaConf.load("src/conf/setup_dataset_nn.yaml")
    cfg.nn.num_epochs = 50
    cfg.nn.early_stopping = False
    cfg.nn.lr = 1e-3
    cfg.nn.optimizer = "LBFGS"
    train_single_pinn(cfg, dataset_name="set5_mixed")


🚀 Training PINN on dataset: data/SM_AVR_GOV/dataset_set5_mixed.pkl
SM_AVR_GOV data
Loading data from: data/SM_AVR_GOV/dataset_set5_mixed.pkl


/Users/jonaswiendl/local/PowerPINN/src/nn/nn_dataset.py:189: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:264.)
  training_sample = torch.tensor(training_sample, dtype=torch.float32) # convert the trajectory to tensor


Number of training samples:  80000 Number of validation samples:  10000 Number of testing samples:  10000
Number of different initial conditions for collocation points:  500
['theta', 'omega', 'E_d_dash', 'E_q_dash', 'R_F', 'V_r', 'E_fd', 'P_sv', 'P_m'] Variables
[[-2, 2], [-1, 1], [0], [0.9, 1.1], [1], [1.105], [1.08], [0.7048], [0.7048]] Set of values for init conditions
[10, 10, 1, 5, 1, 1, 1, 1, 1] Iterations per value
Selected deep learning model:  DynamicNN
Training DynamicNN for 50 epochs (data-only)...
Epoch 10/50 | Train Loss: 8.925e-02 | Val Loss: 6.405e-02
Epoch 20/50 | Train Loss: 4.920e-03 | Val Loss: 4.391e-03
Epoch 30/50 | Train Loss: 2.224e-03 | Val Loss: 2.137e-03
Epoch 40/50 | Train Loss: 1.291e-03 | Val Loss: 1.218e-03
Epoch 50/50 | Train Loss: 7.246e-04 | Val Loss: 7.197e-04
✅ Saved model to: model/SM_AVR_GOV/pinn_set5_mixed_dataonly_test.pth

📉 Final validation loss: 7.197e-04
